# 08 - Seed Study: the statistical-validity gate

Every leaderboard gap on this dataset is currently **single-seed** on a 180-image test set,
so a +0.5 pp difference is inside the noise. This notebook trains each candidate across
**5 seeds** (42/123/777/2025/3407), scores each on TEST, and reports **mean +/- std + 95% CI**
with a **Welch two-sample test** vs the baseline.

**Locked recipe** (identical for every arm): YOLOv8n family, imgsz 640, SGD + cosine LR,
`close_mosaic=10`, `hsv_h=hsv_s=0` (grayscale -> color aug off), `workers=0` (Windows-safe).

**Decision rule:** adopt a candidate over the baseline only if `|t| > 2.78` (df~4) **and** the
95% CIs are disjoint. Otherwise it's a tie -> keep the simpler baseline.

**Cost (RTX 2000 Ada):** ~80 min/run. baseline + P2 = 10 runs (~13 h). Each optional arm
(YOLOv8s, YOLO11n) adds 5 runs. For a faster first read set `SEEDS = [42, 123, 777]` below.
Runs are sequential (one GPU). You launch these; nothing auto-runs.

In [ ]:
import os, sys, subprocess
from pathlib import Path
import torch

ROOT = Path.cwd(); ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
PY = sys.executable

SEEDS   = [42, 123, 777, 2025, 3407]   # <- set to [42, 123, 777] for a faster 3-seed read
SEEDSTR = ",".join(map(str, SEEDS))

print("repo :", Path.cwd())
print("CUDA :", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(CPU only!)")
print("seeds:", SEEDS, f"(~80 min/run; baseline+P2 = {2*len(SEEDS)} runs)")

## Step 1 - Baseline (required): establishes the noise band

In [ ]:
!{PY} scripts/run_seed_study.py --train --tag baseline_640 --model yolov8n.pt --imgsz 640 --epochs 150 --seeds {SEEDSTR}

## Step 2 - P2 head (required): the small-object arm (`configs/yolov8n_p2.yaml`)
Targets `inclusion` (4% median box) + thin `scratches`. ~2.93M params, 12.4 GFLOPs@640.

In [ ]:
!{PY} scripts/run_seed_study.py --train --tag baseline_p2_640 --model configs/yolov8n_p2.yaml --imgsz 640 --epochs 150 --seeds {SEEDSTR}

## Step 3 - OPTIONAL capacity / newer-base arms
`yolov8s.pt` and `yolo11n.pt` auto-download (~25 MB) on first use (needs internet once).
Run these only if you want the full architecture comparison (each = 5 more runs).

In [ ]:
!{PY} scripts/run_seed_study.py --train --tag yolov8s_640 --model yolov8s.pt --imgsz 640 --epochs 150 --seeds {SEEDSTR}

In [ ]:
!{PY} scripts/run_seed_study.py --train --tag yolo11n_640 --model yolo11n.pt --imgsz 640 --epochs 150 --seeds {SEEDSTR}

## Step 4 - Aggregate + compare (CPU, instant)
Compares each trained candidate against the baseline. Skips any arm you didn't train.

In [ ]:
STUDY = ROOT / "experiments" / "seed_study"
candidates = [c for c in ["baseline_p2_640", "yolov8s_640", "yolo11n_640"]
              if any((STUDY / f"{c}_seed{s}" / "test_metrics.json").exists() for s in SEEDS)]
print("comparing baseline_640 vs:", candidates or "(none trained yet)")
for cand in candidates:
    print("=" * 72)
    subprocess.run([PY, "scripts/run_seed_study.py", "--aggregate",
                    "--tag", "baseline_640", "--compare", cand, "--seeds", SEEDSTR])